https://www.kaggle.com/c/bike-sharing-demand/data?select=train.csv

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
%matplotlib inline

이 셀은 **데이터 분석 및 머신러닝 모델 구축에 필요한 주요 라이브러리들을 임포트**합니다. `sklearn`을 통해 모델 선택, 선형 회귀, 평가 지표를 사용하고, `numpy`는 수치 계산, `matplotlib`는 시각화, `pandas`는 데이터 처리를 담당합니다. `%matplotlib inline`은 주피터 노트북에서 그림을 바로 표시하기 위한 설정입니다.

In [3]:
import os
# 노트북 파일이 있는 폴더로 이동 (예시)
os.chdir(r'C:\githome\hipython_rep')

# 변경 후 확인
print("변경 후:", os.getcwd())

변경 후: C:\githome\hipython_rep


데이터 파일의 경로를 설정하기 위해 **현재 작업 디렉토리를 변경하는 코드**입니다. `os.chdir()` 함수를 사용하여 지정된 경로로 이동하며, `os.getcwd()`를 통해 변경된 디렉토리를 확인합니다.

In [4]:
df = pd.read_csv('data1/bike-sharing-demand/train.csv')

캐글에서 제공하는 **자전거 대여량 예측 데이터셋인 'train.csv' 파일을 Pandas DataFrame으로 로드**합니다. 이 데이터는 자전거 대여량 예측 모델 구축의 기반이 됩니다.

In [5]:
test_df = pd.read_csv('data1/bike-sharing-demand/test.csv')

In [6]:
df.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


로드된 데이터프레임 `df`의 **상위 5행을 출력하여 데이터의 초기 구조와 내용을 빠르게 확인**합니다. 각 컬럼의 데이터 형식과 포함된 값들을 엿볼 수 있습니다.

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  object 
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 1020.7+ KB


로드된 데이터프레임 `df`의 **기본 정보를 확인**합니다. 각 컬럼의 이름, Non-Null 값의 개수, 데이터 타입(Dtype), 그리고 메모리 사용량을 통해 데이터의 구조와 누락된 값 여부를 빠르게 파악할 수 있습니다.

In [8]:
datetime_df = df.copy()
datetime_df['datetime'] = pd.to_datetime(datetime_df['datetime'])
datetime_df['year'] = datetime_df['datetime'].dt.year
datetime_df['month'] = datetime_df['datetime'].dt.month
datetime_df['day'] = datetime_df['datetime'].dt.day
datetime_df['hour'] = datetime_df['datetime'].dt.hour

In [34]:
test_datetime_df = test_df.copy()
test_datetime_df['datetime'] = pd.to_datetime(test_datetime_df['datetime'])
test_datetime_df['year'] = test_datetime_df['datetime'].dt.year
test_datetime_df['month'] = test_datetime_df['datetime'].dt.month
test_datetime_df['hour'] = test_datetime_df['datetime'].dt.hour

`datetime` 컬럼을 `pd.to_datetime()`을 통해 **datetime 객체로 변환하고, 여기에서 '년', '월', '일', '시간' 정보를 추출하여 새로운 컬럼으로 추가**합니다. 이러한 시간 기반 파생 변수들은 모델이 시간적 패턴을 학습하는 데 중요한 피처로 활용됩니다.

In [9]:
dropped_df = datetime_df.drop(['datetime','casual','registered'], axis=1)
dropped_df

,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,count,year,month,day,hour
0,1,0,0,1,9.84,14.395,81,0.0000,16,2011,1,1,0
1,1,0,0,1,9.02,13.635,80,0.0000,40,2011,1,1,1
2,1,0,0,1,9.02,13.635,80,0.0000,32,2011,1,1,2
3,1,0,0,1,9.84,14.395,75,0.0000,13,2011,1,1,3
4,1,0,0,1,9.84,14.395,75,0.0000,1,2011,1,1,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10881,4,0,1,1,15.58,19.695,50,26.0027,336,2012,12,19,19
10882,4,0,1,1,14.76,17.425,57,15.0013,241,2012,12,19,20
10883,4,0,1,1,13.94,15.910,61,15.0013,168,2012,12,19,21
10884,4,0,1,1,13.94,17.425,61,6.0032,129,2012,12,19,22


모델 학습에 직접적으로 사용되지 않거나, `count` 컬럼과 중복 정보를 가지는 **'datetime', 'casual', 'registered' 컬럼을 데이터프레임에서 제거**합니다. 이는 모델의 복잡성을 줄이고 예측 성능을 향상시키기 위함입니다. `dropped_df`를 출력하여 변경된 데이터프레임의 모습을 확인합니다.

In [10]:
X = dropped_df.drop('count',axis=1).values
y = dropped_df['count']

예측하고자 하는 **'count' 컬럼을 타겟 변수 `y`로 설정하고, 'count'를 제외한 나머지 모든 컬럼들을 피처(독립 변수) `X`로 분리**합니다. 이는 머신러닝 모델 학습을 위한 표준적인 데이터 준비 과정입니다.

In [11]:
from sklearn.preprocessing import StandardScaler

데이터 스케일링을 위해 `sklearn.preprocessing` 모듈에서 **`StandardScaler`를 임포트**합니다. 이는 피처들의 스케일이 다를 때 모델 성능에 미칠 수 있는 부정적인 영향을 줄여줍니다.


In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

원본 데이터를 **훈련 세트와 테스트 세트로 8:2 비율로 분리**합니다. `random_state=50`은 재현 가능한 결과를 보장합니다. 이후 `StandardScaler`를 사용하여 훈련 세트(`X_train`)에 맞춰 스케일링을 학습하고, 이 스케일링을 훈련 및 테스트 세트(`X_train_scaled`, `X_test_scaled`)에 적용합니다.


In [13]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **선형 회귀 모델(`LinearRegression`)을 초기화하고, 스케일링된 훈련 데이터(`X_train_scaled`, `y_train`)를 사용하여 모델을 학습**시킵니다. 학습된 모델은 `fit()` 메서드를 통해 데이터의 패턴을 파악하고, `predict()` 메서드를 사용하여 스케일링된 테스트 데이터(`X_test_scaled`)에 대한 자전거 대여량 예측값(`y_pred`)을 생성합니다. 이 예측값은 이후 모델의 성능을 평가하는 데 사용됩니다.


In [14]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(141.28875698820917), np.float64(19962.512851273226))

이 셀은 선형 회귀 모델의 **예측 성능을 평가하기 위해 평균 제곱 오차(MSE)와 제곱근 평균 제곱 오차(RMSE)를 계산**합니다. `mean_squared_error` 함수는 실제 값(`y_test`)과 예측 값(`y_pred`) 간의 오차 제곱 평균을 구하고, `np.sqrt()`를 통해 MSE의 제곱근인 RMSE를 계산합니다. RMSE는 예측 오류의 크기를 실제 값과 동일한 단위로 나타내므로 모델의 예측 정확도를 직관적으로 이해하는 데 유용합니다.


In [15]:
r2_score(y_test, y_pred)

np.float64(0.3872731888420481)

이 셀은 선형 회귀 모델의 **결정 계수(R2 스코어)를 계산하여 모델의 설명력을 평가**합니다. `r2_score` 함수는 모델이 종속 변수(`y_test`)의 분산을 얼마나 잘 설명하는지를 0과 1 사이의 값으로 나타냅니다. 1에 가까울수록 모델이 데이터를 잘 설명하며 예측력이 높다고 볼 수 있습니다.

In [16]:
pd.Series(data = np.abs(np.round(lr.coef_,1)), index=dropped_df.drop('count',axis=1).columns).sort_values(ascending=False)

hour          53.3
year          41.5
humidity      38.5
atemp         37.8
month         34.4
temp          15.0
season         8.4
windspeed      6.6
weather        3.5
day            2.8
holiday        0.4
workingday     0.3
dtype: float64

이 셀은 **선형 회귀 모델의 회귀 계수(`lr.coef_`)를 활용하여 각 피처의 중요도를 분석**합니다. 회귀 계수의 절댓값을 사용하여 피처가 타겟 변수(`count`)에 미치는 영향의 크기를 파악하고, 이를 Pandas Series로 변환한 후 내림차순으로 정렬합니다. 이 결과를 통해 어떤 피처(`hour`, `year`, `humidity` 등)가 자전거 대여량 예측에 가장 큰 영향을 미치는지 직관적으로 이해할 수 있습니다. 이는 모델 해석 및 피처 엔지니어링 전략 수립에 중요한 통찰력을 제공합니다.

In [17]:
X = dropped_df.drop(['count', 'workingday', 'holiday', 'weather', 'day'],axis=1).values
y = dropped_df['count']

In [49]:
test_df.head()

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed
0,2011-01-20 00:00:00,1,0,1,1,10.66,11.365,56,26.0027
1,2011-01-20 01:00:00,1,0,1,1,10.66,13.635,56,0.0000
2,2011-01-20 02:00:00,1,0,1,1,10.66,13.635,56,0.0000
3,2011-01-20 03:00:00,1,0,1,1,10.66,12.880,56,11.0014
4,2011-01-20 04:00:00,1,0,1,1,10.66,12.880,56,11.0014


In [50]:
test_X_test = test_datetime_df.drop(['workingday', 'holiday', 'weather', 'datetime'],axis=1).values

이 셀은 **기존 `dropped_df`에서 'count' 뿐만 아니라 'workingday', 'holiday', 'weather', 'day' 컬럼을 추가로 제외하여 피처 `X`를 재설정**합니다. 이는 앞서 수행한 피처 중요도 분석 결과를 바탕으로 모델의 불필요한 복잡성을 줄이고 성능 개선을 시도하기 위함입니다. 타겟 변수 `y`는 동일하게 'count' 컬럼으로 유지됩니다.

In [51]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)
scaler = StandardScaler()
scaler.fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [52]:

test_X_test_scaled = scaler.transform(test_X_test)

이 셀은 **재설정된 피처 `X`와 타겟 `y`를 사용하여 데이터를 다시 훈련 세트와 테스트 세트로 분리하고 스케일링**합니다. `train_test_split`을 통해 8:2 비율로 분리하며 `random_state=50`을 유지하여 일관성을 확보합니다. 이후 `StandardScaler`를 다시 `fit`하고 `transform`하여 새로운 피처 조합에 맞춰 데이터를 표준화합니다. 이 과정은 변경된 피처 셋으로 모델을 재학습하기 위한 필수적인 데이터 전처리입니다.

In [19]:
lr = LinearRegression()
lr.fit(X_train_scaled,y_train)
y_pred = lr.predict(X_test_scaled)

이 셀은 **재조정된 데이터(`X_train_scaled`, `y_train`)를 사용하여 선형 회귀 모델을 다시 학습**하고 예측을 수행합니다. 피처에서 'workingday', 'holiday', 'weather', 'day'가 제외되었으므로, 이전 모델과는 다른 학습 결과를 보일 수 있습니다. 이 단계를 통해 피처 선택이 모델 성능에 미치는 영향을 평가할 수 있습니다.

In [20]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(141.24328004605954), np.float64(19949.6641581696))

이 셀은 피처가 조정된 후 **재학습된 선형 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 다시 계산**합니다. 이전 모델의 평가 결과와 비교하여, 컬럼 제거가 모델의 오차 크기에 어떤 영향을 미쳤는지 확인할 수 있습니다.

In [21]:
r2_score(y_test, y_pred)

np.float64(0.38766756498154453)

이 셀은 피처가 조정된 후 **재학습된 선형 회귀 모델의 결정 계수(R2 스코어)를 계산하여 모델의 설명력을 재평가**합니다. 이전 모델의 R2 스코어와 비교하여, 피처 선택의 변화가 모델의 설명력에 미친 영향을 분석할 수 있습니다.

In [22]:
from sklearn.model_selection import cross_val_score
neg_mse_scores = cross_val_score(lr, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([-10526.25760188, -14643.63675544, -13408.86289528, -32908.22903851,
       -33814.03284637])

이 셀은 **교차 검증(`cross_val_score`)을 사용하여 선형 회귀 모델의 평균 제곱 오차(MSE)를 평가**합니다. `scoring='neg_mean_squared_error'`를 사용하여 음의 MSE 값을 얻고, 이를 통해 모델의 일반화 성능을 여러 폴드에서 확인합니다.

In [23]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(140.93945379974946),
 array([102.59755164, 121.01089519, 115.79664458, 181.40625413,
        183.88592346]))

이 셀은 **교차 검증으로 얻은 음의 MSE 값으로부터 RMSE를 계산**하고, 그 평균과 각 폴드별 RMSE 값을 확인합니다. 이를 통해 모델의 예측 오차 수준을 보다 신뢰성 있게 파악할 수 있습니다.

In [24]:
r2_scores = cross_val_score(lr, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([-0.06216394,  0.34962232,  0.193199  ,  0.29351144,  0.28577092]),
 np.float64(0.21198794795304474))

이 셀은 **교차 검증(`cross_val_score`)을 사용하여 선형 회귀 모델의 결정 계수(R2 스코어)를 평가**합니다. 각 폴드별 R2 스코어와 그 평균을 확인하여 모델의 설명력과 일반화 성능을 종합적으로 판단합니다.

In [25]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=42, max_depth=8)
rf.fit(X_train_scaled,y_train)
y_pred = rf.predict(X_test_scaled)

이 셀은 **앙상블 모델인 `RandomForestRegressor`를 초기화하고 학습**시킵니다. `random_state=42`는 결과의 재현성을 위해 설정되었으며, `max_depth=8`은 트리의 최대 깊이를 제한하여 과적합을 방지합니다. 학습된 모델은 스케일링된 테스트 데이터(`X_test_scaled`)에 대한 자전거 대여량을 예측합니다.

In [26]:
mse = mean_squared_error(y_test,y_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(89.236875925786), np.float64(7963.220024994123))

이 셀은 **랜덤 포레스트 회귀 모델의 예측 성능을 평가하기 위해 MSE와 RMSE를 계산**합니다. 선형 회귀 모델의 결과와 비교하여, 랜덤 포레스트가 자전거 대여량 예측에 있어 어떤 성능을 보이는지 확인할 수 있습니다.

In [27]:
r2_score(y_test, y_pred)

np.float64(0.7555779450805667)

이 셀은 **랜덤 포레스트 회귀 모델의 결정 계수(R2 스코어)를 계산하여 모델의 설명력을 평가**합니다. 선형 회귀 모델의 R2 스코어와 비교하여, 앙상블 모델의 예측력이 얼마나 향상되었는지 파악할 수 있습니다.

In [28]:
neg_mse_scores = cross_val_score(rf, X, y, scoring='neg_mean_squared_error', cv=5)
neg_mse_scores

array([ -9678.12881651,  -6939.06422788, -11682.14633691, -14593.88917869,
       -15238.02031205])

이 셀은 **교차 검증을 통해 랜덤 포레스트 회귀 모델의 평균 제곱 오차(MSE)를 평가**합니다. 여러 폴드에서 얻은 음의 MSE 값을 통해 모델의 안정적인 일반화 성능을 확인합니다.

In [29]:
RMSE = np.sqrt(neg_mse_scores * (-1))
np.mean(RMSE), RMSE

(np.float64(106.80201040211382),
 array([ 98.37748125,  83.30104578, 108.08397817, 120.80517033,
        123.44237648]))

이 셀은 **교차 검증된 음의 MSE 값으로부터 RMSE를 계산**하고, 그 평균과 각 폴드별 RMSE 값을 확인합니다. 이는 랜덤 포레스트 모델의 예측 오차 수준에 대한 보다 신뢰성 있는 지표를 제공합니다.

In [30]:
r2_scores = cross_val_score(rf, X, y, scoring='r2', cv=5)
r2_scores, np.mean(r2_scores)

(array([0.02341745, 0.69181068, 0.29709421, 0.68669187, 0.67813844]),
 np.float64(0.4754305308402337))

이 셀은 **교차 검증을 사용하여 랜덤 포레스트 회귀 모델의 결정 계수(R2 스코어)를 평가**합니다. 각 폴드별 R2 스코어와 그 평균을 확인하여 랜덤 포레스트 모델의 전반적인 설명력과 일반화 성능을 파악합니다.

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures

`sklearn.pipeline`의 `Pipeline`은 여러 처리 단계를 하나로 묶어 워크플로우를 효율화하며, `sklearn.preprocessing`의 `PolynomialFeatures`는 데이터에 다항 특성을 추가하여 모델의 비선형 학습 능력을 높입니다.

In [32]:
results = []
for degree in range(1,6):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('linear',LinearRegression())
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
pd.DataFrame(results)

,degree,MSE,RMSE,R2
0,1,19949.664158,141.243280,0.387668
1,2,15268.674402,123.566478,0.531345
2,3,13195.150231,114.870145,0.594990
3,4,12992.573072,113.984969,0.601208
4,5,14053.712501,118.548355,0.568637


이 셀은 `Pipeline`을 사용하여 **다항 특성 변환(`PolynomialFeatures`)과 선형 회귀 모델(`LinearRegression`)을 결합**하고, **다양한 다항식 차수(1차부터 5차까지)에 따라 모델을 학습하고 성능을 평가**합니다. 각 차수별로 MSE, RMSE, R2 스코어를 계산하여 `results` 리스트에 저장하며, 최종적으로 이를 DataFrame으로 출력하여 **다항 회귀 모델의 차수에 따른 성능 변화를 분석**합니다. 이는 선형 모델이 비선형 관계를 얼마나 잘 표현하는지 확인하는 데 유용합니다.

다항회귀모델(선형)에서는 4차가 가장 높다

In [33]:
results = []
best_r2 = False
for degree in range(1,6):
    model_poly = Pipeline([
        ('poly',PolynomialFeatures(degree=degree, include_bias=False)),
        ('RF',RandomForestRegressor(random_state=22, max_depth=8))
    ])
    model_poly.fit(X_train_scaled, y_train)
    pred_poly = model_poly.predict(X_test_scaled)
    mse = mean_squared_error(y_test,pred_poly)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, pred_poly)
    results.append({
        'degree' : degree,
        'MSE' : mse,
        'RMSE' : rmse,
        'R2' : r2
    })
    if best_r2 < r2 or best_r2 == False:
        best_r2 = r2
        best_model = model_poly
        best_pred_poly = pred_poly
pd.DataFrame(results)

KeyboardInterrupt: 

이 셀은 `Pipeline`을 사용하여 **다항 특성 변환(`PolynomialFeatures`)과 랜덤 포레스트 회귀 모델(`RandomForestRegressor`)을 결합**하고, **다양한 다항식 차수(1차부터 5차까지)에 따라 모델을 학습하고 성능을 평가**합니다. 각 차수별로 MSE, RMSE, R2 스코어를 계산하여 `results` 리스트에 저장합니다. 특히, 이 코드는 **가장 높은 R2 스코어를 기록한 모델(`best_model`)과 해당 예측값(`best_pred_poly`)을 저장**하여 최적의 다항 특성 기반 랜덤 포레스트 모델을 식별하는 데 중점을 둡니다. 최종적으로 `results` DataFrame을 출력하여 **랜덤 포레스트 모델의 차수에 따른 성능 변화 및 최적 모델 선정 결과**를 보여줍니다.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

In [ ]:
gb_clf = GradientBoostingRegressor()
gb_clf.fit(X_train_scaled,y_train)
gb_pred = gb_clf.predict(X_test_scaled)

이 셀은 **그레디언트 부스팅 회귀 모델(`GradientBoostingRegressor`)을 초기화하고 학습**시킵니다. 학습된 모델은 스케일링된 훈련 데이터(`X_train_scaled`, `y_train`)를 사용하여 피팅되며, 이후 테스트 데이터(`X_test_scaled`)에 대한 자전거 대여량 예측(`gb_pred`)을 수행합니다.

In [ ]:
mse = mean_squared_error(y_test,gb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(89.70005003404088), np.float64(8046.098976109437))

이 셀은 **그레디언트 부스팅 회귀 모델의 예측 성능을 평가하기 위해 MSE(평균 제곱 오차)와 RMSE(제곱근 평균 제곱 오차)를 계산**합니다. 이는 모델의 예측 정확도를 나타내는 중요한 지표입니다.

In [ ]:
r2_score(y_test, gb_pred)

np.float64(0.7530340691763986)

이 셀은 **그레디언트 부스팅 회귀 모델의 결정 계수(R2 스코어)를 계산**합니다. R2 스코어는 모델이 종속 변수의 분산을 얼마나 잘 설명하는지를 나타내며, 1에 가까울수록 모델의 설명력이 높다는 것을 의미합니다.

In [53]:
from xgboost import XGBRegressor

In [54]:
xgb = XGBRegressor(n_estimators=400, learning_rate=0.1, max_depth=3, use_label_encoder=False)
evals = [(X_test_scaled, y_test)]
xgb.fit(X_train_scaled, y_train, early_stopping_rounds=40, 
        eval_set=evals, verbose=True)
xgb_pred = xgb.predict(X_test_scaled)

[0]	validation_0-rmse:239.68643
[1]	validation_0-rmse:222.57877
[2]	validation_0-rmse:207.14524
[3]	validation_0-rmse:193.84479
[4]	validation_0-rmse:182.42064
[5]	validation_0-rmse:172.41668
[6]	validation_0-rmse:163.71122
[7]	validation_0-rmse:155.64039
[8]	validation_0-rmse:149.11751
[9]	validation_0-rmse:143.19608
[10]	validation_0-rmse:138.22308
[11]	validation_0-rmse:134.00039
[12]	validation_0-rmse:129.78194
[13]	validation_0-rmse:126.25316
[14]	validation_0-rmse:123.01973
[15]	validation_0-rmse:120.33632
[16]	validation_0-rmse:118.17205
[17]	validation_0-rmse:116.26302
[18]	validation_0-rmse:114.09009
[19]	validation_0-rmse:112.18215
[20]	validation_0-rmse:110.74463
[21]	validation_0-rmse:109.26124
[22]	validation_0-rmse:107.64256
[23]	validation_0-rmse:106.54190
[24]	validation_0-rmse:105.29148
[25]	validation_0-rmse:104.56237
[26]	validation_0-rmse:103.87734
[27]	validation_0-rmse:103.28128
[28]	validation_0-rmse:102.40961
[29]	validation_0-rmse:101.94350
[30]	validation_0-rm

c:\Users\Admin\miniconda3\envs\hi_ml_env\lib\site-packages\xgboost\sklearn.py:793: UserWarning: `early_stopping_rounds` in `fit` method is deprecated for better compatibility with scikit-learn, use `early_stopping_rounds` in constructor or`set_params` instead.
  warnings.warn(


[145]	validation_0-rmse:89.40748
[146]	validation_0-rmse:89.41196
[147]	validation_0-rmse:89.39178
[148]	validation_0-rmse:89.43468
[149]	validation_0-rmse:89.41857
[150]	validation_0-rmse:89.41522
[151]	validation_0-rmse:89.40627
[152]	validation_0-rmse:89.40472
[153]	validation_0-rmse:89.40342
[154]	validation_0-rmse:89.39767
[155]	validation_0-rmse:89.39706
[156]	validation_0-rmse:89.41975
[157]	validation_0-rmse:89.41778
[158]	validation_0-rmse:89.35083
[159]	validation_0-rmse:89.35102
[160]	validation_0-rmse:89.35501
[161]	validation_0-rmse:89.35094
[162]	validation_0-rmse:89.35786
[163]	validation_0-rmse:89.33766
[164]	validation_0-rmse:89.28657
[165]	validation_0-rmse:89.25228
[166]	validation_0-rmse:89.20478
[167]	validation_0-rmse:89.20643
[168]	validation_0-rmse:89.17926
[169]	validation_0-rmse:89.18592
[170]	validation_0-rmse:89.18181
[171]	validation_0-rmse:89.17654
[172]	validation_0-rmse:89.21764
[173]	validation_0-rmse:89.20471
[174]	validation_0-rmse:89.19571
[175]	vali

이 셀은 **XGBoost(`XGBRegressor`) 모델을 초기화하고 학습**시킵니다. `n_estimators`는 부스팅 단계의 수, `learning_rate`는 각 부스팅 단계에서의 가중치 축소율, `max_depth`는 트리의 최대 깊이를 정의합니다. 학습된 모델은 스케일링된 훈련 데이터(`X_train_scaled`, `y_train`)로 피팅되며, 이후 테스트 데이터(`X_test_scaled`)에 대한 자전거 대여량 예측(`xgb_pred`)을 수행합니다.

In [ ]:
mse = mean_squared_error(y_test,xgb_pred)
rmse = np.sqrt(mse)

rmse, mse

(np.float64(88.262751975708), np.float64(7790.313386325347))

이 셀은 **XGBoost 모델의 예측 성능을 평가하기 위해 MSE(평균 제곱 오차)와 RMSE(제곱근 평균 제곱 오차)를 계산**합니다. 이는 모델의 예측 정확도를 나타내는 중요한 지표입니다.

In [ ]:
r2_score(y_test, xgb_pred)

np.float64(0.7608851192889883)

이 셀은 **XGBoost 모델의 결정 계수(R2 스코어)를 계산**합니다. R2 스코어는 모델이 종속 변수의 분산을 얼마나 잘 설명하는지를 나타내며, 1에 가까울수록 모델의 설명력이 높다는 것을 의미합니다.

In [ ]:
from sklearn.linear_model import Ridge, Lasso, ElasticNet

In [ ]:
ridge = Ridge(alpha=10)
ridge.fit(X_train_scaled,y_train)
pred_ridge = ridge.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_ridge)
r2 = r2_score(y_test, pred_ridge)
mse, r2

(np.float64(19950.636490061777), np.float64(0.38763772035105537))

릿지(Ridge) 회귀 모델을 **알파(alpha) 값 10**으로 설정하여 학습하고, 테스트 데이터에 대한 예측을 수행합니다. 이어서 예측 결과와 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [ ]:
from sklearn.linear_model import RidgeCV, LassoCV
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
ridge_cv = RidgeCV(alphas=alphas, cv=5)
ridge_cv.fit(X_train_scaled, y_train)
ridge_preds = ridge_cv.predict(X_test_scaled)
ridge_mse = mean_squared_error(y_test, ridge_preds)
ridge_r2 = r2_score(y_test, ridge_preds)
print(f'ridge cv mse : {ridge_mse:.4f}, r2 : {ridge_r2:.4f}')

ridge cv mse : 19950.6365, r2 : 0.3876


`RidgeCV`를 사용하여 교차 검증을 통해 최적의 알파(alpha) 값을 찾는 릿지(Ridge) 회귀 모델을 학습시킵니다. 이후 테스트 데이터에 대한 예측을 수행하고, 모델의 성능을 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**로 평가하여 출력합니다.

In [ ]:
ridge_cv.alpha_

np.float64(10.0)

`ridge_cv.alpha_`는 `RidgeCV` 모델이 교차 검증을 통해 **자동으로 찾아낸 최적의 알파(alpha) 값**을 나타냅니다. 이 값은 모델 학습 시 가장 좋은 성능을 보인 정규화 강도를 의미합니다.

In [ ]:
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled,y_train)
pred_lasso = lasso.predict(X_test_scaled)

mse = mean_squared_error(y_test, pred_lasso)
r2 = r2_score(y_test, pred_lasso)
mse, r2

(np.float64(19950.156656940588), np.float64(0.387652448287384))

라쏘(Lasso) 회귀 모델을 **알파(alpha) 값 0.1**로 설정하여 학습하고, 테스트 데이터에 대한 예측을 수행합니다. 이어서 예측 결과와 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [ ]:
alphas = [0.001, 0.01, 0.1, 1, 10, 100]
lasso_cv = LassoCV(alphas=alphas, cv=5)
lasso_cv.fit(X_train_scaled, y_train)
lasso_preds = lasso_cv.predict(X_test_scaled)
lasso_mse = mean_squared_error(y_test, lasso_preds)
lasso_r2 = r2_score(y_test, lasso_preds)
print(f'ridge cv mse : {lasso_mse:.4f}, r2 : {lasso_r2:.4f}')

ridge cv mse : 19949.7029, r2 : 0.3877


`LassoCV`를 사용하여 교차 검증을 통해 최적의 알파(alpha) 값을 찾는 라쏘(Lasso) 회귀 모델을 학습시킵니다. 이후 테스트 데이터에 대한 예측을 수행하고, 모델의 성능을 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**로 평가하여 출력합니다.

In [ ]:
lasso_cv.alpha_

np.float64(0.001)

`lasso_cv.alpha_`는 `LassoCV` 모델이 교차 검증을 통해 **자동으로 찾아낸 최적의 알파(alpha) 값**을 나타냅니다. 이 값은 모델 학습 시 가장 좋은 성능을 보인 정규화 강도를 의미합니다.

In [ ]:
lasso_cv.coef_

array([ -8.57149057,  14.69491014,  38.30287494, -40.22357758,
         6.25909402,  41.38964099,  34.8640128 ,  52.90858842])

`lasso_cv.coef_`는 `LassoCV` 모델이 학습 후 **추정한 특성(feature)들의 회귀 계수(coefficients)**를 나타냅니다. 라쏘(Lasso) 회귀의 특성상 이 값들 중 일부는 **0이 되어 특정 특성들이 모델에서 제외**될 수 있습니다.

In [ ]:
ridge_cv.coef_

array([ -7.76999757,  15.51991804,  37.46279412, -40.16682248,
         6.23562866,  41.34664584,  34.04216548,  52.86636101])

`ridge_cv.coef_`는 `RidgeCV` 모델이 학습 후 **추정한 각 특성(feature)들의 회귀 계수(coefficients)**를 나타냅니다.

In [ ]:
enet = ElasticNet(alpha=0.1, l1_ratio=0.5)
enet.fit(X_train_scaled,y_train)

ElasticNet(alpha=0.1)

엘라스틱넷(ElasticNet) 회귀 모델을 **알파(alpha) 값 0.1**과 **L1_ratio 값 0.5**로 설정하여 학습시킵니다.

In [ ]:
enet_pred = enet.predict(X_test_scaled)

NameError: name 'enet' is not defined

학습된 엘라스틱넷(ElasticNet) 모델 `enet`을 사용하여 **스케일링된 테스트 데이터(`X_test_scaled`)에 대한 예측값

In [ ]:
mse = mean_squared_error(y_test, enet_pred)
r2 = r2_score(y_test, enet_pred)
mse, r2

(np.float64(19992.905135231336), np.float64(0.38634033197316586))

엘라스틱넷(ElasticNet) 모델의 예측(`enet_pred`)과 실제 값(`y_test`)을 비교하여 **평균 제곱 오차(MSE)**와 **결정 계수($R^2$)**를 계산하고 출력합니다.

In [ ]:
results = pd.DataFrame({
    '모델' : ['다항회귀', 'XGB', '릿지회귀', '라쏘회귀', '엘라스틱넷회귀'],
    'RMSE' : [np.sqrt(mean_squared_error(y_test,best_pred_poly)),
             np.sqrt(mean_squared_error(y_test,xgb_pred)),
             np.sqrt(mean_squared_error(y_test,pred_ridge)),
             np.sqrt(mean_squared_error(y_test,pred_lasso)),
             np.sqrt(mean_squared_error(y_test,enet_pred))],
    'R2' : [r2_score(y_test,best_pred_poly),
            r2_score(y_test,xgb_pred),
            r2_score(y_test,pred_ridge),
            r2_score(y_test,pred_lasso),
            r2_score(y_test,enet_pred)]
})
results

,모델,RMSE,R2
0,다항회귀,89.190835,0.755830
1,XGB,88.262752,0.760885
2,릿지회귀,141.246722,0.387638
3,라쏘회귀,141.245023,0.387652
4,엘라스틱넷회귀,141.396270,0.386340


여러 회귀 모델(다항회귀, XGB, 릿지회귀, 라쏘회귀, 엘라스틱넷회귀)의 성능 지표인 **RMSE(평균 제곱근 오차)**와 **R2(결정 계수)**를 취합하여 **`results`라는 데이터프레임을 생성하고 출력**합니다.

In [55]:
xgb_test_pred = xgb.predict(test_X_test_scaled)

In [57]:
xgb_test_pred[xgb_test_pred < 0] = 0

In [58]:
submission = pd.read_csv("./data/sampleSubmission.csv")
submission['count'] = xgb_test_pred
submission.to_csv("submission.csv", index=False)
print(submission.head())

              datetime      count
0  2011-01-20 00:00:00  14.160042
1  2011-01-20 01:00:00   5.826628
2  2011-01-20 02:00:00   0.000000
3  2011-01-20 03:00:00   0.000000
4  2011-01-20 04:00:00   0.000000
